In [1]:
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [2]:
env_path = ".env"

if not os.path.exists(env_path):
    env_path = "../.env"

load_dotenv(env_path)

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

print("Database:", db_name)
print("Host:", db_host)
print("Port:", db_port)

Database: car_accidents
Host: localhost
Port: 3306


In [3]:
connection_url = (
    f"mysql+pymysql://{db_user}:{db_password}"
    f"@{db_host}:{db_port}/{db_name}"
)

engine = create_engine(connection_url)

In [4]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT DATABASE();")
    )

    print("Connected database:", result.scalar())

Connected database: car_accidents


In [5]:
csv_path = "../data/processed/final_accidents_weather_air_quality.csv"

accidents_df = pd.read_csv(
    csv_path,
    low_memory=False
)

accidents_df.shape

(74326, 37)

In [6]:
accidents_df.columns = (
    accidents_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

accidents_df = accidents_df.rename(
    columns={
        "date": "accident_date",
        "time": "accident_time"
    }
)

accidents_df.columns

Index(['accident_index', 'longitude', 'latitude', 'accident_severity',
       'number_of_vehicles', 'number_of_casualties', 'accident_date',
       'day_of_week', 'accident_time', 'road_type', 'speed_limit',
       'junction_detail', 'junction_control', 'light_conditions',
       'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area',
       'year', 'severity_label', 'severe_accident', 'accident_datetime',
       'month', 'hour', 'weather_hour', 'grid_latitude', 'grid_longitude',
       'police_force', 'temperature_2m', 'rain', 'snowfall', 'weather_code',
       'cloud_cover', 'wind_speed_10m', 'pm2_5', 'pm10', 'nitrogen_dioxide',
       'european_aqi'],
      dtype='str')

In [7]:
date_columns = [
    "accident_date",
    "accident_datetime",
    "weather_hour"
]

for column in date_columns:
    accidents_df[column] = pd.to_datetime(
        accidents_df[column],
        errors="coerce"
    )

In [8]:
accidents_df[date_columns].dtypes

accident_date        datetime64[us]
accident_datetime    datetime64[us]
weather_hour         datetime64[us]
dtype: object

In [9]:
accidents_df.to_sql(
    name="accidents",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

74326

In [10]:
query = """
SELECT COUNT(*) AS total_rows
FROM accidents;
"""

row_count = pd.read_sql(query, engine)

row_count

,total_rows
0,74326


In [11]:
print("CSV rows:", accidents_df.shape[0])
print("MySQL rows:", row_count.loc[0, "total_rows"])

CSV rows: 74326
MySQL rows: 74326
